In [1]:
#!/usr/bin/env python3
import os
from IPython.display import display
import ipywidgets as widgets
from pathlib import Path
import sys

from lerobot_so_arm.config import get_path
from lerobot_so_arm.utils.visualization import (
    load_trajectory_data_from_episode,
    visualize_trajectory_3d,
    visualize_trajectory_components,
    print_trajectory_statistics,
    list_available_episodes,
    PLOTLY_AVAILABLE
)

DATASET_PATH = get_path('datasets') + '/output_extrinsic'  
WIDGETS_AVAILABLE = True

print(f"Dataset path: {DATASET_PATH}")
print(f"Path exists: {os.path.exists(DATASET_PATH)}")
print(f"Plotly available: {PLOTLY_AVAILABLE}")


Dataset path: /Users/michelmeyer/.local/dev/test_dataset/output_extrinsic
Path exists: True
Plotly available: True


In [2]:
episodes = list_available_episodes(DATASET_PATH)
print(f"Found {len(episodes)} episodes with trajectory data:")
for i, episode in enumerate(episodes):
    print(f"{i+1}. {episode}")

if episodes:
    selected_episode = episodes[0]
    episode_path = os.path.join(DATASET_PATH, selected_episode)
    print(f"\nselecting this episode: {selected_episode}")
    trajectory_data = load_trajectory_data_from_episode(episode_path)
else:
    print("No episodes found with trajectory data")


Found 1 episodes with trajectory data:
1. smanni+train_so100_all-episode_001

selecting this episode: smanni+train_so100_all-episode_001
Found camera extrinsics for: front_camera


In [3]:
# Import the new interactive function
from lerobot_so_arm.utils.visualization import visualize_trajectory_with_robot_interactive

# Load your trajectory data (make sure it includes joint_states)
if trajectory_data:
    print("Creating interactive robot visualization...")
    
    # Create interactive visualization with time step slider
    interactive_viz = visualize_trajectory_with_robot_interactive(
        trajectory_data, 
        title="Robot Trajectory with Interactive Configuration",
        robot_type="so_new_calibration"  # or whatever robot type you're using
    )

Creating interactive robot visualization...
Added 1 camera(s) to visualization


In [4]:
# Create 2D component plots using the new function
if episodes and trajectory_data:
    print("\nCreating 2D component plots...")
    fig_2d = visualize_trajectory_components(
        trajectory_data, 
        title=f"Robot Trajectory Components - {selected_episode}"
    )
    if fig_2d is not None:
        fig_2d.show()
    
    # Interactive episode selector
    if len(episodes) > 1 and WIDGETS_AVAILABLE:
        print("\nInteractive Episode Selector:")
        
        # Create dropdown widget for episode selection
        episode_dropdown = widgets.Dropdown(
            options=episodes,
            value=selected_episode,
            description='Episode:',
            style={'description_width': 'initial'},
            layout={'width': '50%'}
        )
        
        # Create output widget for displaying plots
        output = widgets.Output()
        
        # Define update function
        def on_episode_change(change):
            with output:
                output.clear_output()
                selected_ep = change['new']
                episode_path = os.path.join(DATASET_PATH, selected_ep)
                print(f"Loading trajectory data from: {selected_ep}")
                
                # Load trajectory data using the new function
                traj_data = load_trajectory_data_from_episode(episode_path)
                
                if traj_data:
                    # Print statistics using the new function
                    print_trajectory_statistics(traj_data)
                    
                    # Create 3D visualization using the new function
                    print("\nCreating 3D visualization...")
                    fig_3d = visualize_trajectory_3d(
                        traj_data, 
                        title=f"Robot End-Effector Trajectory - {selected_ep}",
                        show_coord_system=True
                    )
                    
                    if fig_3d is not None:
                        fig_3d.show()
                    
                    # Create 2D component plots using the new function
                    print("Creating 2D component plots...")
                    fig_2d = visualize_trajectory_components(
                        traj_data, 
                        title=f"Robot Trajectory Components - {selected_ep}"
                    )
                    if fig_2d is not None:
                        fig_2d.show()
                else:
                    print("Failed to load trajectory data")
        
        # Register callback
        episode_dropdown.observe(on_episode_change, names='value')
        
        # Display widgets
        display(episode_dropdown)
        display(output)
        
        # Initialize with first episode
        on_episode_change({'new': selected_episode})



Creating 2D component plots...
